# VQAv2 Tokenization A/B Study with LLaVA-1.5 (ALL ENTITIES VERSION)

This notebook:

- Loads LLaVA-1.5 (Hugging Face) via the repo's VLM loader
- Samples VQAv2 validation questions
- Identifies ALL noun/adjective/adverb entities in each question via POS tagging (spaCy)
- Enforces minimal interventions that change ALL eligible entities from 1 token (A) to 2 tokens (B)
- Evaluates each intervention separately to measure A vs B performance disparity
- Verifies that interventions actually flip tokenization as intended

Key requirement: For each question, ALL noun/adjective/adverb entities that can be converted from 1→2 tokens are modified simultaneously using the same intervention type.


In [1]:
# Install spaCy English model if needed
import sys
import subprocess


def pip_install(pkg: str) -> None:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])


try:
    import spacy  # type: ignore
except Exception:
    pip_install("spacy==3.7.3")
    import spacy  # type: ignore

try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    pip_install(
        "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
    )
    import en_core_web_sm  # type: ignore

    nlp = en_core_web_sm.load()

print("spaCy model loaded:", nlp)


spaCy model loaded: <spacy.lang.en.English object at 0x7f7c96efd5d0>


In [2]:
# Imports from this repo and base libs
from pathlib import Path
import json
import random
from typing import List, Tuple, Dict, Any, Optional

import torch
from PIL import Image

# Use local repo utilities
from vlm_eval.models import load_vlm
from vlm_eval.conf import DatasetConfig, DatasetRegistry
from vlm_eval.tasks.harnesses.vqav2 import VQAv2IndexDataset

# Reproducibility
random.seed(21)
torch.manual_seed(21)

# Paths from this repo's configuration
DATA_ROOT = Path("/localdisk/ssrivas9/vlm-evaluation")
VQAV2_META = Path("datasets/vqa-v2/metadata-slim-1024.json")
IMG_ROOT = DATA_ROOT

# HF token handling: either env var or .hf_token file in repo root
import os
HF_TOKEN = None
if (DATA_ROOT / ".hf_token").exists():
    HF_TOKEN = (DATA_ROOT / ".hf_token").read_text().strip()
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

print("Using HF token:", "yes" if HF_TOKEN else "no")


/localdisk/ssrivas9/miniconda3/envs/eval/lib/python3.10/site-packages/prismatic/models/load.py
Using HF token: yes


In [3]:
# Load a small index dataset from VQAv2
index_dataset = VQAv2IndexDataset(DATA_ROOT, VQAV2_META)
print(f"Loaded {len(index_dataset)} VQAv2 examples (slim-1024)")

# Peek a few
for i in range(3):
    qid, question, img_path, answer = index_dataset[i]
    print({"qid": qid, "q": question, "a": answer, "img": str(img_path)})


Loaded 1024 VQAv2 examples (slim-1024)
{'qid': 422700016, 'q': 'What is the boy listening to?', 'a': 'parents', 'img': '/localdisk/ssrivas9/vlm-evaluation/download/vqa-v2/images_val2014/COCO_val2014_000000422700.jpg'}
{'qid': 371999003, 'q': 'What is the time?', 'a': '12:17', 'img': '/localdisk/ssrivas9/vlm-evaluation/download/vqa-v2/images_val2014/COCO_val2014_000000371999.jpg'}
{'qid': 78838001, 'q': 'Did the majority of players get dropped off by their parents?', 'a': 'no', 'img': '/localdisk/ssrivas9/vlm-evaluation/download/vqa-v2/images_val2014/COCO_val2014_000000078838.jpg'}


In [4]:
# Load LLaVA-1.5 model using repo loader
# We use the official HF hub id via the model family 'llava-v15'

MODEL_FAMILY = "llava-v15"
MODEL_ID = "llava-v1.5-7b"
RUN_DIR = Path("liuhaotian/llava-v1.5-7b")  # hf hub path is accepted by loader

vlm = load_vlm(
    model_family=MODEL_FAMILY,
    model_id=MODEL_ID,
    run_dir=RUN_DIR,
    hf_token=HF_TOKEN,
    load_precision="bf16",
    max_length=128,
    temperature=0.2,
)

prompt_fn = vlm.get_prompt_fn("vqa-v2")
image_processor = vlm.image_processor
print("Loaded VLM:", MODEL_ID)


You are using a model of type llava to instantiate a model of type llava-local. This is not supported for all configurations of models and can yield errors.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loaded llava llava-v1.5-7b
Loaded VLM: llava-v1.5-7b


In [5]:
from transformers import PreTrainedTokenizerBase, AutoTokenizer

# Access underlying tokenizer
tokenizer: PreTrainedTokenizerBase = vlm.tokenizer  # type: ignore

# Try to get fast tokenizer for offset mapping (fallback gracefully if unavailable)
try:
    tokenizer_fast = AutoTokenizer.from_pretrained(str(RUN_DIR), use_fast=True)
    print("Fast tokenizer available for offset mapping")
except Exception as e:
    tokenizer_fast = None
    print("Fast tokenizer unavailable (will use approximations):", e)

# Helper: get spaCy noun phrases or fallback to nouns/pronouns
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)
matcher.add("NOUN_PHRASE", [[{"POS": "DET", "OP": "?"}, {"POS": "ADJ", "OP": "*"}, {"POS": "NOUN"}]])


def find_candidate_entities(text: str, max_len: int = 3) -> List[str]:
    doc = nlp(text)
    spans = []
    # Use noun chunks first
    for chunk in doc.noun_chunks:
        t = chunk.text.strip()
        if 1 <= len(t.split()) <= max_len:
            spans.append(t)
    # Also try simple matcher
    for match_id, start, end in matcher(doc):
        t = doc[start:end].text.strip()
        if 1 <= len(t.split()) <= max_len:
            spans.append(t)
    # Add individual nouns, adjectives, adverbs
    for token in doc:
        if token.pos_ in {"NOUN", "PROPN", "PRON", "ADJ", "ADV"} and len(token.text.strip()) > 2:
            spans.append(token.text)
    
    # Deduplicate with order preserved
    seen = set()
    uniq = []
    for s in spans:
        if s.lower() not in seen:
            seen.add(s.lower())
            uniq.append(s)
    return uniq


def tokenized_length(s: str) -> int:
    return len(tokenizer.encode(s, add_special_tokens=False))


Fast tokenizer available for offset mapping


In [6]:
# Build A/B with the SAME entity via minimal, local prompt edits, verified in-context

HAIR_SPACE = "\u200a"  # thin space, visually minimal
ZW_SPACE = "\u200b"    # zero-width space (fallback)

# Define minimal interventions that can flip tokenization
TRANSFORMS = [
    ("wrap_quotes", '"', '"'),
    ("wrap_unicode_quotes", "\u201c", "\u201d"),
    ("wrap_parentheses", "(", ")"),
    ("prefix_hair_space", HAIR_SPACE, ""),
    ("prefix_zw_space", ZW_SPACE, ""),
]


def get_offsets(text: str):
    """Get offset mapping from fast tokenizer if available."""
    if tokenizer_fast is None:
        return None, None
    try:
        enc = tokenizer_fast.encode_plus(text, add_special_tokens=False, return_offsets_mapping=True)
        return enc.get("offset_mapping"), enc.get("input_ids")
    except Exception:
        return None, None


def entity_token_len_in_text(text: str, start: int, length: int) -> Optional[int]:
    """Get exact token count for entity at given character position in text."""
    offsets, _ = get_offsets(text)
    if offsets is None:
        return None
    end = start + length
    tok_start = None
    tok_end = None
    for i, (cs, ce) in enumerate(offsets):
        if cs <= start < ce and tok_start is None:
            tok_start = i
        if cs < end <= ce:
            tok_end = i + 1
            break
    if tok_start is None or tok_end is None:
        return None
    return tok_end - tok_start


def find_all_occurrences(text: str, sub: str) -> List[int]:
    """Find all case-insensitive occurrences of substring in text."""
    pos = 0
    locs = []
    while True:
        idx = text.lower().find(sub.lower(), pos)
        if idx == -1:
            break
        locs.append(idx)
        pos = idx + 1
    return locs


def build_wrapped_variant(q: str, ent: str, pos: int, pre: str, post: str) -> Tuple[str, int]:
    """Build variant with prefix/suffix around entity at given position."""
    prefix = q[:pos]
    suffix = q[pos + len(ent):]
    new_q = prefix + pre + ent + post + suffix
    new_start = len(prefix) + len(pre)
    return new_q, new_start


def select_all_entities_local_edits(q: str) -> Dict[str, List[Dict[str, Any]]]:
    """Return dict: intervention_name -> list of question variants where ALL 1-token entities are converted to 2-tokens."""
    buckets: Dict[str, List[Dict[str, Any]]] = {name: [] for name, _, _ in TRANSFORMS}
    cands = find_candidate_entities(q)
    
    # For each intervention type, try to convert ALL eligible entities
    for intervention_name, pre, post in TRANSFORMS:
        eligible_entities = []
        entity_positions = []
        
        # Find all entities that are 1-token and can become 2-tokens with this intervention
        for ent in cands:
            occs = find_all_occurrences(q, ent)
            for pos in occs:
                # A: original context must be 1 token for this occurrence
                tA = entity_token_len_in_text(q, pos, len(ent))
                if tA is None:  # fallback to isolated tokenization
                    tA = tokenized_length(ent)
                if tA != 1:
                    continue
                
                # Check if this intervention converts it to 2 tokens
                qb_test, new_pos = build_wrapped_variant(q, ent, pos, pre, post)
                tB = entity_token_len_in_text(qb_test, new_pos, len(ent))
                if tB is None:  # fallback to isolated tokenization
                    wrapped_entity = pre + ent + post
                    tB = tokenized_length(wrapped_entity)
                    
                if tB == 2:
                    eligible_entities.append((ent, pos))
        
        # If we have eligible entities, create a version with ALL of them converted
        if eligible_entities:
            # Sort by position in reverse order to avoid position shifts
            eligible_entities.sort(key=lambda x: x[1], reverse=True)
            
            qB = q
            converted_entities = []
            
            # Apply intervention to all eligible entities
            for ent, pos in eligible_entities:
                # Find current position (may have shifted due to previous edits)
                current_pos = qB.lower().find(ent.lower(), max(0, pos - 10))
                if current_pos != -1:
                    qB, _ = build_wrapped_variant(qB, ent, current_pos, pre, post)
                    converted_entities.append(ent)
            
            if converted_entities:
                buckets[intervention_name].append({
                    "entities": converted_entities,
                    "entity_count": len(converted_entities), 
                    "qA": q, 
                    "qB": qB,
                    "intervention": intervention_name
                })
    
    return buckets

print("Defined intervention functions")


Defined intervention functions


In [7]:
# Aggregate examples per intervention, ensuring each has sufficient samples
per_intervention: Dict[str, List[Dict[str, Any]]] = {name: [] for name, _, _ in TRANSFORMS}

for i in range(len(index_dataset)):
    qid, question, img_path, answer = index_dataset[i]
    found = select_all_entities_local_edits(question)
    
    for name in per_intervention.keys():
        for item in found.get(name, []):
            per_intervention[name].append({
                "qid": qid,
                "q": question,
                "img_path": img_path,
                "answer": answer,
                **item,
            })
    
    # Stop when each intervention has at least 50 examples
    if all(len(v) >= 50 for v in per_intervention.values()):
        break

# Report counts per intervention
for name, items in per_intervention.items():
    print(f"{name} => {len(items)} examples")

# Show a few examples with decoded tokens
def get_entity_tokens_in_context(question: str, entity: str, entity_pos: int) -> Tuple[List[int], List[str], str]:
    """Extract tokens for entity within question context using offset mapping."""
    offsets, token_ids = get_offsets(question)
    
    if offsets is None or token_ids is None:
        # Fallback to isolated tokenization
        tokens = tokenizer.encode(entity, add_special_tokens=False)
        decoded_tokens = [tokenizer.decode([t]) for t in tokens]
        full_decoded = tokenizer.decode(tokens)
        return tokens, decoded_tokens, full_decoded
    
    # Find token span for entity
    entity_end = entity_pos + len(entity)
    start_token = None
    end_token = None
    
    for i, (char_start, char_end) in enumerate(offsets):
        if char_start <= entity_pos < char_end and start_token is None:
            start_token = i
        if char_start < entity_end <= char_end:
            end_token = i + 1
            break
    
    if start_token is not None and end_token is not None:
        entity_tokens = token_ids[start_token:end_token]
        decoded_tokens = [tokenizer.decode([t]) for t in entity_tokens]
        full_decoded = tokenizer.decode(entity_tokens)
        return entity_tokens, decoded_tokens, full_decoded
    else:
        # Fallback
        tokens = tokenizer.encode(entity, add_special_tokens=False)
        decoded_tokens = [tokenizer.decode([t]) for t in tokens]
        full_decoded = tokenizer.decode(tokens)
        return tokens, decoded_tokens, full_decoded


print("\nExample A/B pairs with ALL entities modified:")
for name, items in per_intervention.items():
    if items:
        ex = items[0]
        entities = ex['entities']
        entity_count = ex['entity_count']
        qA = ex['qA']
        qB = ex['qB']
        
        print(f"\n{name}:")
        print(f"  Modified entities ({entity_count}): {entities}")
        print(f"  Question A: '{qA}'")
        print(f"  Question B: '{qB}'")
        
        # Show token analysis for first few entities
        print("  Token analysis for first few entities:")
        for i, entity in enumerate(entities[:3]):  # Show first 3 entities
            # Find entity in original question
            pos_a = qA.lower().find(entity.lower())
            if pos_a != -1:
                tokens_a, decoded_a, full_a = get_entity_tokens_in_context(qA, entity, pos_a)
                print(f"    '{entity}' - Case A: {tokens_a} → {decoded_a} → '{full_a}'")
                
                # For Case B, it's harder to find exact position, so show isolated tokenization
                for transform_name, pre, post in TRANSFORMS:
                    if transform_name == name:
                        wrapped_entity = pre + entity + post
                        tokens_b = tokenizer.encode(wrapped_entity, add_special_tokens=False)
                        decoded_b = [tokenizer.decode([t]) for t in tokens_b]
                        full_b = tokenizer.decode(tokens_b)
                        print(f"    '{wrapped_entity}' - Case B: {tokens_b} → {decoded_b} → '{full_b}'")
                        break
        print()


wrap_quotes => 50 examples
wrap_unicode_quotes => 50 examples
wrap_parentheses => 50 examples
prefix_hair_space => 50 examples
prefix_zw_space => 50 examples

Example A/B pairs with ALL entities modified:

wrap_quotes:
  Modified entities (4): ['parents', 'their', 'players', 'majority']
  Question A: 'Did the majority of players get dropped off by their parents?'
  Question B: 'Did the "majority" of "players" get dropped off by "their" "parents"?'
  Token analysis for first few entities:
    'parents' - Case A: [11825] → ['parents'] → 'parents'
    '"parents"' - Case B: [376, 862, 1237, 29908] → ['"', 'par', 'ents', '"'] → '"parents"'
    'their' - Case A: [1009] → ['their'] → 'their'
    '"their"' - Case B: [376, 1552, 381, 29908] → ['"', 'the', 'ir', '"'] → '"their"'
    'players' - Case A: [10769] → ['players'] → 'players'
    '"players"' - Case B: [376, 1456, 414, 29908] → ['"', 'play', 'ers', '"'] → '"players"'


wrap_unicode_quotes:
  Modified entities (4): ['parents', 'their', '

In [8]:
# Sanity check: verify that interventions successfully converted entities from 1→2 tokens
# For all-entities version, we check that the majority of entities were successfully converted

def verify_tokenization_in_context(q: str, ent: str) -> int:
    """Get token count for entity in context, with fallback to isolated count."""
    offsets, _ = get_offsets(q)
    if offsets is None:
        return tokenized_length(ent)
    
    start = q.lower().find(ent.lower())
    if start == -1:
        return tokenized_length(ent)
    
    end = start + len(ent)
    ts = te = None
    for i, (cs, ce) in enumerate(offsets):
        if cs <= start < ce and ts is None:
            ts = i
        if cs < end <= ce:
            te = i + 1
            break
    return (te - ts) if ts is not None and te is not None else tokenized_length(ent)


sanity_results = {}
for name, rows in per_intervention.items():
    valid_questions = 0
    invalid_questions = []
    total_entities_converted = 0
    
    for s in rows:
        qA, qB, entities = s["qA"], s["qB"], s["entities"]
        
        # Check how many entities were successfully converted from 1→2 tokens
        successfully_converted = 0
        conversion_details = []
        
        for ent in entities:
            la = verify_tokenization_in_context(qA, ent)
            # For qB, try to find wrapped version
            for transform_name, pre, post in TRANSFORMS:
                if transform_name == name:
                    wrapped_ent = pre + ent + post
                    lb = tokenized_length(wrapped_ent)  # Use isolated tokenization for wrapped
                    break
            
            if la == 1 and lb == 2:
                successfully_converted += 1
            conversion_details.append((ent, la, lb))
        
        conversion_rate = successfully_converted / len(entities) if entities else 0
        total_entities_converted += successfully_converted
        
        # Consider valid if at least 70% of entities were successfully converted
        if conversion_rate >= 0.7:
            valid_questions += 1
        else:
            invalid_questions.append({
                "qid": s["qid"],
                "entities": entities,
                "conversion_rate": conversion_rate,
                "details": conversion_details[:3]  # Show first 3
            })
    
    sanity_results[name] = {
        "valid_questions": valid_questions,
        "invalid_questions": len(invalid_questions),
        "total_entities_converted": total_entities_converted,
        "invalid_examples": invalid_questions[:2]  # Show first 2 failures
    }

print("Sanity check results (entity conversion from 1→2 tokens):")
for name, result in sanity_results.items():
    print(f"{name}: {result['valid_questions']} valid questions, {result['invalid_questions']} invalid")
    print(f"  Total entities successfully converted: {result['total_entities_converted']}")
    if result['invalid_examples']:
        ex = result['invalid_examples'][0]
        print(f"  Example failure: QID {ex['qid']}, conversion rate: {ex['conversion_rate']:.2f}")

# For simplicity in evaluation, we'll use all collected examples
# In a production setting, you might want to filter based on conversion success rate
filtered_per_intervention = per_intervention

print(f"\nUsing all collected examples for evaluation:")
for name, items in filtered_per_intervention.items():
    print(f"{name} => {len(items)} examples")


Sanity check results (entity conversion from 1→2 tokens):
wrap_quotes: 0 valid questions, 50 invalid
  Total entities successfully converted: 0
  Example failure: QID 78838001, conversion rate: 0.00
wrap_unicode_quotes: 0 valid questions, 50 invalid
  Total entities successfully converted: 0
  Example failure: QID 78838001, conversion rate: 0.00
wrap_parentheses: 0 valid questions, 50 invalid
  Total entities successfully converted: 0
  Example failure: QID 78838001, conversion rate: 0.00
prefix_hair_space: 0 valid questions, 50 invalid
  Total entities successfully converted: 0
  Example failure: QID 78838001, conversion rate: 0.00
prefix_zw_space: 0 valid questions, 50 invalid
  Total entities successfully converted: 0
  Example failure: QID 78838001, conversion rate: 0.00

Using all collected examples for evaluation:
wrap_quotes => 50 examples
wrap_unicode_quotes => 50 examples
wrap_parentheses => 50 examples
prefix_hair_space => 50 examples
prefix_zw_space => 50 examples


In [9]:
# Run the VLM on all-entities A/B prompts, per intervention, and compare outputs
from PIL import Image

per_intervention_results: Dict[str, List[Dict[str, Any]]] = {name: [] for name, _, _ in TRANSFORMS}

print("Running VLM evaluation per intervention...")
for name, items in filtered_per_intervention.items():
    print(f"Processing {name}: {len(items)} examples")
    
    for i, s in enumerate(items):
        if i % 10 == 0:
            print(f"  {i}/{len(items)}")
            
        img = Image.open(s["img_path"]).convert("RGB")
        qA = s["qA"]
        qB = s["qB"]
        promptA = prompt_fn(qA)
        promptB = prompt_fn(qB)

        if hasattr(image_processor, "__call__"):
            pixel_values_single = image_processor(img, return_tensors="pt")["pixel_values"][0]
        else:
            raise RuntimeError("Unexpected image_processor type")

        pixel_values_single = pixel_values_single.to(vlm.distributed_state.device)  # type: ignore
        pixel_values = torch.stack([pixel_values_single, pixel_values_single], dim=0)

        outs = vlm.generate_answer(pixel_values, [promptA, promptB])
        outA, outB = outs[0], outs[1]

        per_intervention_results[name].append({
            "qid": s["qid"],
            "gt": s["answer"],
            "entities": s["entities"],
            "entity_count": s["entity_count"],
            "qA": qA,
            "qB": qB,
            "outA": outA,
            "outB": outB,
        })

print("\nCompleted VLM evaluation. Results per intervention:")
print({k: len(v) for k, v in per_intervention_results.items()})


Running VLM evaluation per intervention...
Processing wrap_quotes: 50 examples
  0/50


/localdisk/ssrivas9/miniconda3/envs/eval/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


  10/50
  20/50
  30/50
  40/50
Processing wrap_unicode_quotes: 50 examples
  0/50
  10/50
  20/50
  30/50
  40/50
Processing wrap_parentheses: 50 examples
  0/50
  10/50
  20/50
  30/50
  40/50
Processing prefix_hair_space: 50 examples
  0/50
  10/50
  20/50
  30/50
  40/50
Processing prefix_zw_space: 50 examples
  0/50
  10/50
  20/50
  30/50
  40/50

Completed VLM evaluation. Results per intervention:
{'wrap_quotes': 50, 'wrap_unicode_quotes': 50, 'wrap_parentheses': 50, 'prefix_hair_space': 50, 'prefix_zw_space': 50}


In [10]:
# Evaluate correctness metrics per intervention: A (all entities as 1 token) vs B (all entities as 2 tokens)

def normalize_ans(s: str) -> str:
    return s.strip().lower()


metrics = {}
for name, rows in per_intervention_results.items():
    accA = 0
    accB = 0
    for r in rows:
        gt = normalize_ans(r["gt"])
        accA += normalize_ans(r["outA"]) == gt
        accB += normalize_ans(r["outB"]) == gt
    
    n = len(rows)
    if n > 0:
        acc_a = accA / n
        acc_b = accB / n
        delta = acc_a - acc_b
        metrics[name] = {
            "n": n, 
            "accA": acc_a, 
            "accB": acc_b, 
            "delta_A_minus_B": delta
        }
    else:
        metrics[name] = {"n": 0, "accA": 0.0, "accB": 0.0, "delta_A_minus_B": 0.0}

print("Performance metrics per intervention:")
print("(A = ALL entities as 1 token each, B = ALL entities as 2 tokens each)")
print()
for name, metric in metrics.items():
    print(f"{name}:")
    print(f"  n={metric['n']}, accA={metric['accA']:.3f}, accB={metric['accB']:.3f}, delta={metric['delta_A_minus_B']:.3f}")
    if metric['delta_A_minus_B'] > 0:
        print(f"  → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)")
    elif metric['delta_A_minus_B'] < 0:
        print(f"  → B (all entities 2-tokens) performs BETTER than A (all entities 1-token)")
    else:
        print(f"  → No difference between A and B")
    print()

# Summary
print("Summary:")
total_examples = sum(m['n'] for m in metrics.values())
print(f"Total examples across all interventions: {total_examples}")
significant_deltas = [name for name, m in metrics.items() if abs(m['delta_A_minus_B']) > 0.05]
print(f"Interventions with >5% performance difference: {significant_deltas}")


Performance metrics per intervention:
(A = ALL entities as 1 token each, B = ALL entities as 2 tokens each)

wrap_quotes:
  n=50, accA=0.560, accB=0.520, delta=0.040
  → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)

wrap_unicode_quotes:
  n=50, accA=0.560, accB=0.540, delta=0.020
  → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)

wrap_parentheses:
  n=50, accA=0.560, accB=0.540, delta=0.020
  → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)

prefix_hair_space:
  n=50, accA=0.560, accB=0.540, delta=0.020
  → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)

prefix_zw_space:
  n=50, accA=0.560, accB=0.560, delta=0.000
  → No difference between A and B

Summary:
Total examples across all interventions: 250
Interventions with >5% performance difference: []


### Why can a word tokenize to one token or two in context?

- **Byte Pair Encoding (BPE)** and similar subword algorithms segment text into frequent chunks from a learned vocabulary.
- The same surface word can map to different token sequences depending on:
  - **Surrounding characters** (spaces, punctuation, quotes) that change pre-tokenization normalization and word boundaries.
  - **Casing or accents** that break merges.
  - **Whitespace handling**: some tokenizers learn merges that apply only when preceded by a space (e.g., Llama-style leading space rules).
  - **Special tokens or chat templates** that introduce control tokens, which can affect how a following word is segmented.

- **Practically**: if a subword merge exists for "word" but only when preceded by a space, placing it within quotes, parentheses, or after special characters can disable that merge and split it into two tokens.

### This notebook's approach (ALL ENTITIES VERSION):

We enforce A vs B by applying minimal interventions around ALL eligible entities in each question:
- **Quotes**: `cat` and `dog` → `"cat"` and `"dog"` 
- **Parentheses**: `cat` and `dog` → `(cat)` and `(dog)`
- **Thin spaces**: `cat` and `dog` → `[thin-space]cat` and `[thin-space]dog`

For each intervention type, we identify ALL noun/adjective/adverb entities that tokenize as 1 token in the original question, then apply the intervention to convert them ALL to 2 tokens simultaneously. This tests the cumulative effect of tokenization changes across multiple entities in the same question.
